[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Boolean Filtering &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Count the hours Tromso was warmer than 10 degrees.

In [2]:
tromso_warm = (readings["station"] == "Tromso") & (readings["celsius"] > 10)

print(tromso_warm.sum())

1847


The two conditions combine with `&`, and `sum` counts the rows where both are `True`.

**2.** Show Svalbard's readings above 5 degrees, and print how many there are.

In [3]:
svalbard_mild = readings[(readings["station"] == "Svalbard") & (readings["celsius"] > 5)]

print(len(svalbard_mild), "readings")
svalbard_mild

711 readings


,station,time,celsius
14462,Svalbard,2025-05-31 15:00:00,5.1
14554,Svalbard,2025-06-01 14:00:00,5.2
14646,Svalbard,2025-06-02 13:00:00,5.2
14754,Svalbard,2025-06-03 16:00:00,5.2
14758,Svalbard,2025-06-03 17:00:00,5.2
...,...,...,...
23390,Svalbard,2025-09-01 15:00:00,5.4
23394,Svalbard,2025-09-01 16:00:00,5.6
23482,Svalbard,2025-09-02 14:00:00,5.3
23486,Svalbard,2025-09-02 15:00:00,5.7


The filtered table keeps each row's index label, its time and its reading, and `len` counts its
rows.

**3.** Count the hours at Bergen or Oslo below -5 degrees, using `isin` and `&`, and say which
station they came from.

In [4]:
cold_west_and_east = readings[readings["station"].isin(["Bergen", "Oslo"]) & (readings["celsius"] < -5)]

print(cold_west_and_east["station"].value_counts())
print("Bergen's coldest hour:", readings.loc[readings["station"] == "Bergen", "celsius"].min())

station
Oslo    134
Name: count, dtype: int64
Bergen's coldest hour: -4.8


Every one of them is Oslo's. Bergen never fell below -5 degrees, since its coldest hour of the year
was -4.8.

**4.** Count each station's hours between 0 and 1 degree, both ends included, with `between` and
`value_counts`.

In [5]:
readings[readings["celsius"].between(0, 1)]["station"].value_counts()

station
Oslo        575
Svalbard    492
Bergen      440
Tromso      393
Name: count, dtype: int64

`between` includes both ends by default, so readings of exactly 0 and exactly 1 are counted.

**5.** Use `query` to find Oslo's hours above 18 degrees, and print how many there are.

In [6]:
oslo_hot = readings.query("station == 'Oslo' and celsius > 18")

print(len(oslo_hot), "hours")

137 hours


Inside `query`, the condition is text, so `and` is allowed and the column names need no
`readings[...]` around them.

---

&#8592; **Back to:** [Boolean Filtering](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/08-boolean-filtering.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
